In [2]:
import pandas as pd
import ast

# ============================================
# STEP 1: Load raw data
# ============================================
raw_path = r"D:\SUBJECTS\Data_Analyst\AnalystProjects\datanalystproject\raw_jobs_final.csv"
df = pd.read_csv(raw_path)
print("Starting rows:", len(df))

# ============================================
# STEP 2: remove Duplicates (only check real job_id pe)
# ============================================
df_with_id = df[df["job_id"].notnull()].drop_duplicates(subset="job_id")
df_without_id = df[df["job_id"].isnull()]
df = pd.concat([df_with_id, df_without_id], ignore_index=True)
print("After removing duplicates:", len(df))

# ============================================
# STEP 3: skill_tags text convert into real list  
# ============================================
def safe_convert(text):
    try:
        return ast.literal_eval(text)
    except:
        return []

df["skill_tags"] = df["skill_tags"].apply(safe_convert)

# ============================================
# STEP 4: only valid technical skills  (curated whitelist)
# ============================================
valid_skills = [
    "SQL", "Python", "Power BI", "Excel", "Tableau", "R", "VBA", "Vlookup",
    "Machine Learning", "Statistics", "AWS", "Azure", "Data Visualization",
    "ETL", "MySQL", "PostgreSQL", "Data Analysis", "Data Cleaning",
    "Google Analytics", "Data Management", "Data Mining", "Pyspark",
    "Numpy", "Pandas", "NoSQL", "Big Data", "Google Data Studio",
    "Business Metrics", "Campaign Analytics", "MS Office", "Data Reporting"
]

def filter_valid_skills(tags):
    return [tag for tag in tags if any(valid.lower() == tag.lower().strip() for valid in valid_skills)]

df["clean_skills"] = df["skill_tags"].apply(filter_valid_skills)
print("Jobs with at least 1 valid skill:", df["clean_skills"].apply(len).gt(0).sum())

# ============================================
# STEP 5: break Experience into min/max numbers 
# ============================================
def extract_min_exp(exp_text):
    try:
        return int(str(exp_text).split("-")[0].strip())
    except:
        return None

def extract_max_exp(exp_text):
    try:
        return int(str(exp_text).split("-")[1].replace("Yrs", "").strip())
    except:
        return None

df["exp_min"] = df["experience"].apply(extract_min_exp)
df["exp_max"] = df["experience"].apply(extract_max_exp)

# keep Experience  buckets in group  (for Donut chart)
def experience_bucket(exp_min):
    if pd.isna(exp_min):
        return "Not Specified"
    elif exp_min <= 1:
        return "Fresher (0-1 yr)"
    elif exp_min <= 3:
        return "Junior (1-3 yr)"
    else:
        return "Senior (3+ yr)"

df["experience_bucket"] = df["exp_min"].apply(experience_bucket)

# ============================================
# STEP 6: clean the Location
# ============================================
df["primary_location"] = df["location"].apply(lambda x: str(x).split(",")[0].strip() if pd.notnull(x) else None)

# ============================================
# STEP 7: break Salary into min/max numbers (Lacs PA)
# ============================================
def extract_min_salary(sal_text):
    try:
        sal_text = str(sal_text).replace("Lacs PA", "").strip()
        return float(sal_text.split("-")[0].strip())
    except:
        return None

def extract_max_salary(sal_text):
    try:
        sal_text = str(sal_text).replace("Lacs PA", "").strip()
        return float(sal_text.split("-")[1].strip())
    except:
        return None

df["salary_min"] = df["salary"].apply(extract_min_salary)
df["salary_max"] = df["salary"].apply(extract_max_salary)
df["salary_avg"] = (df["salary_min"] + df["salary_max"]) / 2

# ============================================
# STEP 8:convert  Rating into number 
# ============================================
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# ============================================
# STEP 9: Company tier classify  (MNC vs Others) — optional
# ============================================
known_mncs = ["TCS", "Infosys", "Wipro", "Deloitte", "Accenture", "Cognizant", "IBM", "Capgemini", "HCL"]

def classify_company_tier(company_name):
    if pd.isna(company_name):
        return "Unknown"
    for mnc in known_mncs:
        if mnc.lower() in str(company_name).lower():
            return "MNC"
    return "Other"

df["company_tier"] = df["company"].apply(classify_company_tier)

# ============================================
# STEP 10: remove  Missing critical data  rows 
# ============================================
df = df.dropna(subset=["title"])
print("Final cleaned rows:", len(df))

# ===================================
# STEP 11: Save
# ===================================
save_path = r"D:\SUBJECTS\Data_Analyst\AnalystProjects\datanalystproject\cleaned_jobs_final.csv"
df.to_csv(save_path, index=False)
print("Cleaned file saved at:", save_path)

# ============================================
# Preview & Missing Values Check
# ============================================
print("\n--- Data Preview ---")
print(df[["title", "company", "rating", "salary_min", "salary_max", "experience_bucket", "primary_location"]].head(10))

print("\n--- Missing Values Check ---")
print(df.isnull().sum())

Starting rows: 299
After removing duplicates: 299
Jobs with at least 1 valid skill: 270
Final cleaned rows: 299
Cleaned file saved at: D:\SUBJECTS\Data_Analyst\AnalystProjects\datanalystproject\cleaned_jobs_final.csv

--- Data Preview ---
                                               title  \
0                                       Data Analyst   
1                                       Data Analyst   
2  Looking For immediate joiners - Lab Data Analy...   
3  Data Analyst - 10th April - Virtual Interview ...   
4                                       Data Analyst   
5  Auth Fraud Supervisor- Data Analyst, UIDAI/Ben...   
6                                       Data Analyst   
7                                       Data Analyst   
8                                       Data Analyst   
9                                       Data Analyst   

                                          company  rating  salary_min  \
0                               Wonderla Holidays     4.4         NaN  